In [0]:
%sql
SELECT *
FROM healthcare_dev.bronze.visits_raw;

In [0]:
# ============================================================
# SILVER LAYER — FACT_VISIT
# Source: healthcare_dev.bronze.visits_raw
# Target: healthcare_dev.silver.fact_visit
# ============================================================

from pyspark.sql.functions import (
    col,
    lag,
    to_date,
    datediff,
    current_timestamp
)
from delta.tables import DeltaTable

# ------------------------------------------------------------
# 1. Configure tables and checkpoint
# ------------------------------------------------------------

bronze_table = "healthcare_dev.bronze.visits_raw"
silver_table = "healthcare_dev.silver.fact_visit"

silver_patient_table = "healthcare_dev.silver.dim_patient"
silver_hospital_table = "healthcare_dev.silver.dim_hospital"
silver_diagnosis_table = "healthcare_dev.silver.dim_diagnosis"

checkpoint_path = (
    "abfss://healthcare@ecommerceunifiedproject.dfs.core.windows.net/"
    "silver/fact_visit/checkpoint/"
)

# ------------------------------------------------------------
# 2. Read Silver dimensions
# ------------------------------------------------------------

df_patient = spark.read.table(silver_patient_table)
df_hospital = spark.read.table(silver_hospital_table)
df_diagnosis = spark.read.table(silver_diagnosis_table)

# ------------------------------------------------------------
# 3. Read Bronze visits using Structured Streaming
# ------------------------------------------------------------

df_visit_bronze = spark.readStream.table(bronze_table)

# ------------------------------------------------------------
# 4. Rename duplicate columns
# ------------------------------------------------------------

df_patient = df_patient.withColumnRenamed(
    "city", "patient_city"
)

df_hospital = df_hospital.withColumnRenamed(
    "city", "hospital_city"
)

# ------------------------------------------------------------
# 5. Join visits with Silver dimensions
# ------------------------------------------------------------

df_fact_combined = (
    df_visit_bronze
    .join(df_patient, "patient_id", "left")
    .join(df_hospital, "hospital_id", "left")
    .join(df_diagnosis, "diagnosis_code", "left")
    .withColumn("admission_date", to_date("admission_date"))
    .withColumn("discharge_date", to_date("discharge_date"))
    .withColumn("load_timestamp", current_timestamp())
)

# ------------------------------------------------------------
# 6. Define Delta MERGE
# ------------------------------------------------------------

def merge_fact_visit(batch_df, batch_id):

    if not spark.catalog.tableExists(silver_table):
        (
            batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )
        return

    fact = DeltaTable.forName(spark, silver_table)

    (
        fact.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.visit_id = s.visit_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# ------------------------------------------------------------
# 7. Execute streaming ingestion
# ------------------------------------------------------------

query = (
    df_fact_combined
    .drop("load_timestamp")
    .writeStream
    .foreachBatch(merge_fact_visit)
    .outputMode("update")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .start()
)

query.awaitTermination()

print("Silver fact_visit ingestion completed.")

# ------------------------------------------------------------
# 8. Verify Silver fact table
# ------------------------------------------------------------

display(
    spark.table("healthcare_dev.silver.fact_visit")
)

In [0]:
%sql
SELECT 'dim_diagnosis' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT diagnosis_code) AS unique_keys
FROM healthcare_dev.silver.dim_diagnosis

UNION ALL

SELECT 'dim_hospital',
       COUNT(*),
       COUNT(DISTINCT hospital_id)
FROM healthcare_dev.silver.dim_hospital

UNION ALL

SELECT 'dim_patient',
       COUNT(*),
       COUNT(DISTINCT patient_id)
FROM healthcare_dev.silver.dim_patient

UNION ALL

SELECT 'fact_visit',
       COUNT(*),
       COUNT(DISTINCT visit_id)
FROM healthcare_dev.silver.fact_visit;